# PatchPilot Phase 8: Trace Distillation & 4-bit QLoRA Fine-Tuning (Colab T4)

This standalone notebook fine-tunes `Qwen/Qwen2.5-Coder-7B-Instruct` on the `seeded_train` dataset (`distillation_train.jsonl`, 360 cases, zero `seeded_eval` overlap) using 4-bit QLoRA (`r=16`, `lora_alpha=32`) on a free Google Colab / Kaggle T4 GPU (16 GB VRAM).

After training, it exports a quantized `q4_k_m` GGUF checkpoint and an Ollama `Modelfile` so you can register the distilled model in local host Ollama (`ollama create patchpilot-qwen2.5-coder:7b-qlora -f Modelfile`) and evaluate it with `python -m patchpilot.distillation compare`.

In [ ]:
# 1. Install Unsloth, TRL, PEFT, and BitsAndBytes for 4-bit QLoRA on a Colab T4 GPU
import subprocess
import sys

PACKAGES = [
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
    "trl>=0.12.0",
    "peft>=0.13.0",
    "accelerate>=1.0.0",
    "bitsandbytes>=0.44.0",
    "datasets>=3.0.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", *PACKAGES])


In [ ]:
# 2. Mount Google Drive or locate uploaded `distillation_train.jsonl` and verify zero eval leakage
import json
from pathlib import Path

DATA_PATH = Path("distillation_train.jsonl")
if not DATA_PATH.is_file():
    try:
        from google.colab import drive  # type: ignore[import-not-found]

        drive.mount("/content/drive")
        drive_candidate = Path("/content/drive/MyDrive/patchpilot/distillation_train.jsonl")
        if drive_candidate.is_file():
            DATA_PATH = drive_candidate
    except ImportError:
        pass

assert DATA_PATH.is_file(), f"Upload distillation_train.jsonl first (looked at {DATA_PATH})"

samples = [
    json.loads(line)
    for line in DATA_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print(f"Loaded {len(samples)} training samples from {DATA_PATH}")
for item in samples:
    assert item.get("pool") == "seeded_train", f"Purity violation: non-train pool {item.get('pool')}"
print("Purity check passed: 100% of samples originate from seeded_train.")


In [ ]:
# 3. Load 4-bit base model and attach QLoRA adapters (r=16, lora_alpha=32, attention + MLP modules)
import torch
from unsloth import FastLanguageModel  # type: ignore[import-not-found]

BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_SEQ_LENGTH = 2048
LORA_RANK = 16
LORA_ALPHA = 32
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=1337,
)


In [ ]:
# 4. Format ChatML messages and run SFTTrainer on T4 GPU
from datasets import Dataset  # type: ignore[import-not-found]
from transformers import TrainingArguments  # type: ignore[import-not-found]
from trl import SFTTrainer  # type: ignore[import-not-found]

def format_chat_batch(batch: dict[str, list[object]]) -> dict[str, list[str]]:
    texts: list[str] = []
    for msgs in batch["messages"]:
        rendered = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(rendered)
    return {"text": texts}

raw_dataset = Dataset.from_list(samples)
train_dataset = raw_dataset.map(format_chat_batch, batched=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=1337,
        output_dir="outputs_patchpilot_qlora",
        report_to="none",
    ),
)

train_stats = trainer.train()
print("Training finished:", train_stats.metrics)


In [ ]:
# 5. Export quantized GGUF (q4_k_m) and generate Ollama Modelfile for host registration
GGUF_DIR = Path("patchpilot_qwen25_coder_7b_gguf")
model.save_pretrained_gguf(
    str(GGUF_DIR),
    tokenizer,
    quantization_method="q4_k_m",
)

MODElFILE_CONTENT = '''FROM ./unsloth.Q4_K_M.gguf
PARAMETER temperature 0.0
PARAMETER num_ctx 4096
SYSTEM """You are PatchPilot, an autonomous automated code repair agent. Output ONLY raw SEARCH/REPLACE blocks."""
'''

modelfile_path = GGUF_DIR / "Modelfile"
modelfile_path.write_text(MODElFILE_CONTENT, encoding="utf-8")
print(f"Wrote GGUF and Ollama Modelfile to {modelfile_path}")
print("On the Windows host run: ollama create patchpilot-qwen2.5-coder:7b-qlora -f Modelfile")
